# Twin-Prime-Compatible Residue Transition Experiment

**Prime Numbers Lab**

This notebook tracks the residue transitions modulo \(30\) that are compatible with twin primes:

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1.
\]

It produces:

```text
figures/twin_prime_transition_entries.png
figures/twin_prime_transition_entries.csv
```

The notebook is designed to work inside the repo, but also includes fallback functions so it can run in Colab.

## 1. Setup

In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

TWIN_TRANSITIONS = [
    (11, 13),
    (17, 19),
    (29, 1),
]

print("Figure directory:", FIG_DIR.resolve())
print("Data directory:", DATA_DIR.resolve())

## 2. Load repo functions if available

If `src/transitions.py` exists, this notebook uses it.

Otherwise, it defines local fallback functions.

In [ ]:

try:
    from src.transitions import primes_mod_30, build_transition_matrix, RESIDUES
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}
    print("Loaded transition utilities from src.transitions")
except Exception as e:
    print("Using local fallback transition utilities:", repr(e))

    RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

    def primes_mod_30(primes):
        return np.array([int(p) % 30 for p in primes if int(p) > 5])

    def build_transition_matrix(seq):
        n = len(RESIDUES)
        P = np.zeros((n, n), dtype=float)

        for i in range(len(seq) - 1):
            a, b = int(seq[i]), int(seq[i + 1])
            if a in RES_IDX and b in RES_IDX:
                P[RES_IDX[a], RES_IDX[b]] += 1

        row_sums = P.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return P / row_sums

## 3. Load primes

Preferred input:

```text
data/primes.npy
```

If that file is missing, the notebook falls back to generating primes with `sympy`.

In [ ]:

def load_primes(path=DATA_DIR / "primes.npy", fallback_limit=2_000_000):
    path = Path(path)

    if path.exists():
        print(f"Loading primes from {path}")
        return np.load(path)

    print(f"{path} not found.")
    print(f"Generating primes up to {fallback_limit:,} using sympy fallback...")

    try:
        from sympy import primerange
    except ImportError as exc:
        raise ImportError(
            "sympy is required for fallback prime generation. "
            "Install with `pip install sympy`, or provide data/primes.npy."
        ) from exc

    primes = np.array(list(primerange(2, fallback_limit)), dtype=np.int64)
    print(f"Generated {len(primes):,} primes.")
    return primes

primes = load_primes()
len(primes), primes[:10], primes[-5:]

## 4. Compute transition entries across sample sizes

In [ ]:

def sample_sizes(n_total):
    candidates = [
        1_000,
        3_000,
        10_000,
        30_000,
        100_000,
        300_000,
        1_000_000,
        3_000_000,
        10_000_000,
    ]
    sizes = [n for n in candidates if n <= n_total]
    if not sizes:
        sizes = [n_total]
    return sizes

def transition_entry(P, src, dst):
    return P[RES_IDX[int(src)], RES_IDX[int(dst)]]

sizes = sample_sizes(len(primes))
print("Sample sizes:", sizes)

results = {f"{a}->{b}": [] for a, b in TWIN_TRANSITIONS}

for n in sizes:
    seq = primes_mod_30(primes[:n])
    P = build_transition_matrix(seq)

    for a, b in TWIN_TRANSITIONS:
        results[f"{a}->{b}"].append(transition_entry(P, a, b))

df = pd.DataFrame({"N": sizes, **results})
df

## 5. Plot twin-prime-compatible transition entries

In [ ]:

plt.figure(figsize=(8, 5))

for label in results:
    plt.plot(df["N"], df[label], marker="o", label=label)

plt.xscale("log")
plt.xlabel("Number of primes used")
plt.ylabel("Transition probability")
plt.title("Twin-prime-compatible residue transitions modulo 30")
plt.legend(title="Transition")
plt.tight_layout()

png_path = FIG_DIR / "twin_prime_transition_entries.png"
plt.savefig(png_path, dpi=220)
plt.show()

print("Saved:", png_path)

## 6. Save data

In [ ]:

csv_path = FIG_DIR / "twin_prime_transition_entries.csv"
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df

## 7. Optional: inspect final transition matrix

This displays the first-order transition operator \(P\) for the largest sample size used.

In [ ]:

seq = primes_mod_30(primes[:sizes[-1]])
P = build_transition_matrix(seq)

plt.figure(figsize=(6, 5))
plt.imshow(P, cmap="viridis")
plt.colorbar(label="transition probability")
plt.xticks(range(len(RESIDUES)), RESIDUES)
plt.yticks(range(len(RESIDUES)), RESIDUES)
plt.xlabel("Next residue")
plt.ylabel("Current residue")
plt.title(f"Transition operator P, N={sizes[-1]:,}")
plt.tight_layout()
plt.show()

## 8. Interpretation

The three tracked entries correspond to twin-prime-compatible residue transitions modulo \(30\):

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1.
\]

This experiment does **not** prove the twin prime conjecture.

It gives a direct way to measure how those residue transitions appear in the transition operator \(P\), and how they change as the number of primes \(N\) increases.

In [ ]:

# Optional Colab download cell
# from google.colab import files
# files.download(str(FIG_DIR / "twin_prime_transition_entries.png"))
# files.download(str(FIG_DIR / "twin_prime_transition_entries.csv"))